# RecruitGen — Hybrid GAN + LLM: role-specific synthetic candidate profiles

**Why:** the original GAN (`GAN_Training.ipynb`) learns from 9 features extracted from the Kaggle Resume Dataset. Its
output is limited by that data, not by the GAN: the 24 Kaggle categories are not the 5 RecruitGen roles, 84% of real
rows have experience 0 (the extractor only reads "N years of experience" phrases), 91% have 0 certifications while the
rest are often misreads ("50 certificates" → clipped to 15), and skill scores are coarse. The Kaggle dataset has almost no
resumes for the 5 roles (AI/ML Engineer 0, Cloud Engineer 1 by job title).

**Hybrid pipeline (5 roles: AI/ML Engineer, Cloud Engineer, Data Analyst, General IT, Software Engineer):**

```
LLM structured suggestions ──► resume text ──► app feature extractor ──► validation + de-duplication ──► seed data
seed train split ──► conditional WGAN-GP ──► GAN samples ──► plausibility / duplicate / preference filters ──► profiles
                                                   (optional) LLM description ──► re-extraction check ──► kept or dropped
```

* The LLM never supplies the numbers directly and can never bypass validation; feature values of generated profiles always
  come from the GAN. Sources are labelled (`GAN`, `AI-assisted seed (validated)`, `AI-assisted (validated)`).
* API settings are read from the app's existing `.streamlit/secrets.toml` `[ai]` section or `AI_*` environment variables;
  responses are cached in `data/llm_cache/`, rate limits / failures fall back gracefully.
* **Evaluation honesty:** the 5-role training and test data are LLM-generated (synthetic). The only real 5-role check is
  the small set of Kaggle resumes whose own job title matches a role (weak labels).

Code: `models/gan/llm_assist.py`, `hybrid.py`, `hybrid_train.py`. Outputs: `models/gan/hybrid/`.

## 1. Setup

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)                      # so .streamlit/secrets.toml (API settings) is found

from models.gan import hybrid as hy
from models.gan import hybrid_train
from models.gan.dataset import FEATURE_NAMES
from models.gan.llm_assist import LLMClient

QUICK_RUN = False                   # True = 3 LLM calls per role, 300 GAN iterations, 2 seeds
STAGE = "all"                       # "all" = build/extend seed (cached) + train + evaluate · "train" = reuse saved seed, no API calls
print("AI service configured:", LLMClient().available())

## 2. Real-data patterns used as hints (Kaggle resumes whose job title matches a role)

In [ ]:
real = hy.kaggle_title_subset(with_text=True)
print(real["job_role"].value_counts())
for role, hint in hy.real_pattern_hints(real).items():
    print(f"\n{role}: {hint}")

## 3. Run the pipeline — seed data · GAN training · filters · evaluation

In [ ]:
metrics = hybrid_train.run(quick=QUICK_RUN, stage=STAGE)

## 4. Seed dataset — validated LLM suggestions

In [ ]:
seed = hy.load_seed()
rep = metrics["seed_dataset"]
print(f"kept {rep['kept']} of {rep['requested']} suggested profiles · API calls {metrics['llm_calls']}")
print("rejected:", json.dumps(rep["rejected"], indent=1))
seed.groupby("job_role")[FEATURE_NAMES].agg(["mean", "std"]).round(2).T

In [ ]:
seed[["profile_id", "job_role", "title", "experience_years", "projects_count", "certification_count",
      "education_level", "skills", "summary"]].sample(8, random_state=0)

## 5. Original vs improved output

In [ ]:
pd.DataFrame(metrics["original_vs_improved"]).T.round(4)

In [ ]:
from models.gan.gan import CandidateGAN, MODEL_DIR
hyb = hy.HybridGenerator()
orig = CandidateGAN(MODEL_DIR)
orig_vals, _ = orig.generate_array({"Information Technology": 500}, seed=1)
hyb_vals = np.array([[r[f] for f in FEATURE_NAMES] for role in hy.ROLES for r in hyb.generate(role, 100, seed=1)[0]])
fig, ax = plt.subplots(3, 3, figsize=(15, 10))
for j, (a, f) in enumerate(zip(ax.flat, FEATURE_NAMES)):
    bins = np.linspace(*hy.FEATURE_BOUNDS[f], 31)
    a.hist(orig_vals[:, j], bins=bins, alpha=0.5, density=True, label="original GAN (Kaggle IT)")
    a.hist(hyb_vals[:, j], bins=bins, alpha=0.5, density=True, label="hybrid (5 roles)")
    a.hist(seed[f], bins=bins, histtype="step", lw=2, density=True, label="validated seed")
    a.set_title(f); a.legend(fontsize=7)
plt.suptitle("Feature distributions: original vs improved"); plt.tight_layout(); plt.show()

## 6. Quality, diversity and similarity (held-out seed test split)

In [ ]:
pd.DataFrame([{"generator": n, **{k: v for k, v in q["quality"].items() if isinstance(v, float)},
               **q["diversity"]} for n, q in metrics["quality"].items()]).set_index("generator").round(4).T

## 7. Downstream classifier — seed only vs augmented (untouched test split) and real check

In [ ]:
aug = metrics["augmentation"]
rows = [{"training data": "seed only", "strategy": "-", **{k: f"{v['mean']:.4f} ± {v['std']:.4f}" for k, v in aug["real_only"]["test"].items()}}]
for name, s in aug["samplers"].items():
    c = s["strategies"][s["chosen_strategy"]]
    rows.append({"training data": f"seed + {name}", "strategy": c["label"],
                 **{k: f"{v['mean']:.4f} ± {v['std']:.4f}" for k, v in c["test"].items()},
                 "Δ macro-F1": f"{s['test_f1_macro_delta_vs_real']['mean']:+.4f}"})
pd.DataFrame(rows).set_index("training data")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
d = metrics["detailed_test_report"]
for key in ("real_only", "augmented"):
    print(f"===== {key} (seed test split) =====\n" + d[key]["classification_report_text"])
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for a, key in zip(ax, ("real_only", "augmented")):
    ConfusionMatrixDisplay(np.array(d[key]["confusion_matrix"]), display_labels=hy.ROLES).plot(ax=a, xticks_rotation=30, cmap="Purples", colorbar=False)
    a.set_title(f"{key} · accuracy {d[key]['accuracy']:.3f}")
plt.tight_layout(); plt.show()

In [ ]:
rc = metrics["real_check"]
if rc:
    first = next(iter(rc.values()))
    print(f"Real check: {first['rows']} Kaggle resumes labelled by job title {first['per_role']} · majority-class accuracy {first['majority_class_accuracy']:.3f}")
    display(pd.DataFrame({n: {k: f"{v[k]['mean']:.4f} ± {v[k]['std']:.4f}" for k in ("accuracy", "precision_macro", "recall_macro", "f1_macro")} for n, v in rc.items()}).T)
    print(rc["Seed + hybrid"]["detail_seed0"]["classification_report_text"])

## 8. Example: GAN profiles with validated AI-assisted descriptions

In [ ]:
rows, stats = hyb.generate("Cloud Engineer", 5, seed=3, experience_range=(3, 12), min_scores={"cloud_score": 45})
descriptions, summary = hy.describe_rows("Cloud Engineer", rows)
print(stats, summary)
out = pd.DataFrame(rows)
out["description_source"] = [hy.DESCRIPTION_SOURCE if i in descriptions else "-" for i in range(len(rows))]
out["skills"] = [descriptions.get(i, {}).get("skills", "") for i in range(len(rows))]
out

## 9. Limitations

* The 5-role seed data is LLM-generated; "realistic" means consistent with the app's feature extractor and the hints from
  real resumes, not drawn from a real population. Accuracy on the seed test split measures consistency with that data.
* The real check uses weak labels (job titles) and only roles that exist in the Kaggle data (mostly General IT).
* The feature extractor is keyword-based (`ai`, `ml`, `s3` also match inside other words) and is shared with Transformer V3,
  so it was left unchanged.